In [1]:
import duckdb
import pandas as pd
from pathlib import Path

PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
con = duckdb.connect()

con.execute(f"""
    CREATE VIEW ratings_raw AS SELECT * FROM read_parquet('{PARQUET_DIR}/ratings.parquet');
    CREATE VIEW movies      AS SELECT * FROM read_parquet('{PARQUET_DIR}/movies.parquet');
""")

# identify bot/suspicious users: very low std + many ratings, OR extreme mean
suspicious = con.execute("""
    WITH user_stats AS (
        SELECT 
            userId,
            COUNT(*) AS n,
            AVG(rating) AS mean_r,
            STDDEV(rating) AS std_r
        FROM ratings_raw
        GROUP BY userId
    )
    SELECT *
    FROM user_stats
    WHERE (std_r < 0.3 AND n >= 50)
       OR (mean_r < 1.0 AND n >= 100)
       OR (mean_r > 4.8 AND n >= 100)
""").fetchdf()

print(f"suspicious users found: {len(suspicious):,}")
print(f"total ratings from them: {suspicious['n'].sum():,}")
print(f"as % of all ratings: {suspicious['n'].sum() / 25_000_095 * 100:.2f}%")
print()
print("sample of suspicious users:")
print(suspicious.head(10))

suspicious users found: 276
total ratings from them: 50,611
as % of all ratings: 0.20%

sample of suspicious users:
   userId     n    mean_r     std_r
0   31383    82  4.804878  0.245403
1   61667    76  4.986842  0.080568
2   64231    91  4.873626  0.230861
3   25424   493  4.998986  0.022519
4  112321  1332  3.501502  0.043313
5  113848   166  4.852410  0.564796
6   49263    69  4.985507  0.084497
7  135960   109  4.871560  0.453401
8   40145    60  5.000000  0.000000
9  105670   343  4.845481  0.379714


In [2]:
con.execute("""
    CREATE OR REPLACE VIEW suspicious_users AS
    WITH user_stats AS (
        SELECT 
            userId,
            COUNT(*) AS n,
            AVG(rating) AS mean_r,
            STDDEV(rating) AS std_r
        FROM ratings_raw
        GROUP BY userId
    )
    SELECT userId
    FROM user_stats
    WHERE (std_r < 0.3 AND n >= 50)
       OR (mean_r < 1.0 AND n >= 100)
       OR (mean_r > 4.8 AND n >= 100);
""")

# create cleaned view, downcast to smaller dtypes
clean_path = PARQUET_DIR / "ratings_clean.parquet"

con.execute(f"""
    COPY (
        SELECT 
            CAST(userId AS INTEGER)    AS userId,
            CAST(movieId AS INTEGER)   AS movieId,
            CAST(rating AS REAL)       AS rating,
            CAST(timestamp AS INTEGER) AS timestamp
        FROM ratings_raw
        WHERE userId NOT IN (SELECT userId FROM suspicious_users)
    ) TO '{clean_path}' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# verify
con.execute(f"CREATE VIEW ratings AS SELECT * FROM read_parquet('{clean_path}');")
result = con.execute("""
    SELECT 
        COUNT(*) AS rows,
        COUNT(DISTINCT userId) AS users,
        COUNT(DISTINCT movieId) AS movies,
        ROUND(AVG(rating), 4) AS mean_rating
    FROM ratings
""").fetchdf()
print("cleaned ratings:")
print(result)
print()

# file size
size_mb = clean_path.stat().st_size / 1e6
print(f"file size: {size_mb:.1f} mb (zstd compression)")

cleaned ratings:
       rows   users  movies  mean_rating
0  24949484  162265   58616       3.5316

file size: 147.5 mb (zstd compression)


In [3]:
q = """
WITH user_features AS (
    SELECT 
        userId,
        COUNT(*) AS num_ratings,
        ROUND(AVG(rating), 4) AS mean_rating,
        ROUND(STDDEV(rating), 4) AS std_rating,
        MIN(rating) AS min_rating,
        MAX(rating) AS max_rating,
        COUNT(DISTINCT movieId) AS distinct_movies,
        MIN(timestamp) AS first_ts,
        MAX(timestamp) AS last_ts,
        ROUND((MAX(timestamp) - MIN(timestamp)) / 86400.0, 1) AS active_days,
        ROUND(100.0 * SUM(CASE WHEN rating >= 4.0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_high,
        ROUND(100.0 * SUM(CASE WHEN rating <= 2.0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_low
    FROM ratings
    GROUP BY userId
)
SELECT * FROM user_features
ORDER BY num_ratings DESC
LIMIT 10
"""
con.execute(q).fetchdf()

,userId,num_ratings,mean_rating,std_rating,min_rating,max_rating,distinct_movies,first_ts,last_ts,active_days,pct_high,pct_low
0,72315,32202,3.0806,0.7447,0.5,5.0,32202,1450162767,1570529429,1393.1,14.70,10.81
1,80974,9178,3.2803,0.5602,0.5,5.0,9178,997035038,1450989889,5254.1,25.87,5.18
2,137293,8913,3.1840,0.5218,0.5,5.0,8913,1239072608,1573869979,3875.0,9.91,4.96
3,33844,7919,2.5801,1.0574,0.5,5.0,7919,1355365302,1572798415,2516.6,11.01,36.73
4,20055,7488,3.2089,0.9479,1.0,5.0,7488,1160462492,1423722957,3047.0,33.32,17.98
5,109731,6647,2.8167,0.9227,0.5,5.0,6647,1025845021,1574175899,6346.4,17.39,26.91
6,92046,6564,3.4752,0.9977,0.5,5.0,6564,1000134901,1574135194,6643.5,44.33,13.12
7,49403,6553,1.5226,1.2405,0.5,5.0,6553,955113196,1573791090,7160.6,9.25,76.93
8,30879,5693,2.8765,0.6951,0.5,5.0,5693,1462783923,1522955141,696.4,8.66,16.46
9,115102,5649,2.4628,1.4295,0.5,5.0,5649,1466830526,1518528108,598.4,21.79,43.16


In [4]:
features_path = PARQUET_DIR / "user_features.parquet"

con.execute(f"""
    COPY (
        SELECT 
            userId,
            COUNT(*) AS num_ratings,
            CAST(AVG(rating) AS REAL) AS mean_rating,
            CAST(COALESCE(STDDEV(rating), 0) AS REAL) AS std_rating,
            CAST(MIN(rating) AS REAL) AS min_rating,
            CAST(MAX(rating) AS REAL) AS max_rating,
            CAST(MAX(timestamp) - MIN(timestamp) AS INTEGER) AS active_seconds,
            CAST(100.0 * SUM(CASE WHEN rating >= 4.0 THEN 1 ELSE 0 END) / COUNT(*) AS REAL) AS pct_high,
            CAST(100.0 * SUM(CASE WHEN rating <= 2.0 THEN 1 ELSE 0 END) / COUNT(*) AS REAL) AS pct_low,
            CAST(MIN(timestamp) AS INTEGER) AS first_rating_ts,
            CAST(MAX(timestamp) AS INTEGER) AS last_rating_ts
        FROM ratings
        GROUP BY userId
    ) TO '{features_path}' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# verify
uf = pd.read_parquet(features_path)
print(f"saved: {features_path}")
print(f"shape: {uf.shape}")
print(f"size:  {features_path.stat().st_size / 1e6:.2f} mb")
print(f"\ndtypes:\n{uf.dtypes}")
print(f"\nstats:\n{uf.describe().round(2)}")

saved: /Users/nitishpatil/projects/recsys/data/parquet/user_features.parquet
shape: (162265, 11)
size:  4.33 mb

dtypes:
userId               int32
num_ratings          int64
mean_rating        float32
std_rating         float32
min_rating         float32
max_rating         float32
active_seconds       int32
pct_high           float32
pct_low            float32
first_rating_ts      int32
last_rating_ts       int32
dtype: object

stats:
          userId  num_ratings  mean_rating  std_rating  min_rating  \
count  162265.00    162265.00    162265.00   162265.00   162265.00   
mean    81279.25       153.76         3.68        0.93        1.29   
std     46922.90       267.83         0.48        0.26        0.81   
min         1.00        20.00         0.50        0.00        0.50   
25%     40641.00        36.00         3.40        0.75        0.50   
50%     81279.00        71.00         3.70        0.91        1.00   
75%    121918.00       162.00         4.00        1.09        2.00   


In [5]:
movie_features_path = PARQUET_DIR / "movie_features.parquet"

con.execute(f"""
    COPY (
        SELECT 
            r.movieId,
            COUNT(*) AS num_ratings,
            COUNT(DISTINCT r.userId) AS num_unique_users,
            CAST(AVG(r.rating) AS REAL) AS mean_rating,
            CAST(COALESCE(STDDEV(r.rating), 0) AS REAL) AS std_rating,
            CAST(MIN(r.timestamp) AS INTEGER) AS first_rating_ts,
            CAST(MAX(r.timestamp) AS INTEGER) AS last_rating_ts,
            CAST(100.0 * SUM(CASE WHEN r.rating >= 4.0 THEN 1 ELSE 0 END) / COUNT(*) AS REAL) AS pct_high,
            CAST(100.0 * SUM(CASE WHEN r.rating <= 2.0 THEN 1 ELSE 0 END) / COUNT(*) AS REAL) AS pct_low,
            -- bayesian smoothed mean: pull toward global mean for low-rating-count movies
            -- formula: (n * movie_mean + C * global_mean) / (n + C), C = 25
            CAST((COUNT(*) * AVG(r.rating) + 25.0 * 3.5316) / (COUNT(*) + 25.0) AS REAL) AS smoothed_mean
        FROM ratings r
        GROUP BY r.movieId
    ) TO '{movie_features_path}' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

mf = pd.read_parquet(movie_features_path)
print(f"saved: {movie_features_path}")
print(f"shape: {mf.shape}")
print(f"size:  {movie_features_path.stat().st_size / 1e6:.2f} mb")
print()

# show why smoothing matters
print("comparison of raw mean vs smoothed mean for low-rating-count movies:")
sample = con.execute(f"""
    SELECT 
        m.title,
        mf.num_ratings,
        ROUND(mf.mean_rating, 3) AS raw_mean,
        ROUND(mf.smoothed_mean, 3) AS smoothed_mean,
        ROUND(mf.mean_rating - mf.smoothed_mean, 3) AS shrinkage
    FROM read_parquet('{movie_features_path}') mf
    JOIN movies m ON m.movieId = mf.movieId
    WHERE mf.num_ratings <= 5 AND mf.mean_rating = 5.0
    LIMIT 10
""").fetchdf()
print(sample)

saved: /Users/nitishpatil/projects/recsys/data/parquet/movie_features.parquet
shape: (58616, 10)
size:  1.34 mb

comparison of raw mean vs smoothed mean for low-rating-count movies:
                                               title  num_ratings  raw_mean  \
0  Hijacking Catastrophe: 9/11, Fear & the Sellin...            1       5.0   
1                         Always a Bridesmaid (2000)            1       5.0   
2                             Latin Music USA (2009)            1       5.0   
3         Eye for an Eye, An (Silmä silmästä) (1999)            1       5.0   
4                              Parasites, Les (1999)            1       5.0   
5                     Iran Is Not the Problem (2008)            1       5.0   
6                               Only Daughter (2013)            1       5.0   
7  Pursuit of Unhappiness, The (Anleitung zum Ung...            1       5.0   
8                             Woman's Tale, A (1991)            1       5.0   
9  The Secret Country: The F

what's the smoothing trick? simple but important.

if a movie has 1 rating of 5.0, is its "true" mean rating 5.0? no. it's somewhere between 5.0 and the global mean (3.53), heavily pulled toward the global mean because we have so little data.

formula: (n × movie_mean + C × global_mean) / (n + C)
where C is the "prior strength." we picked C = 25, meaning the prior counts as 25 hypothetical ratings at the global mean. for a movie with 1 rating, smoothed_mean ≈ (5.0 + 25*3.53) / 26 = 3.59. for a movie with 1000 ratings, smoothed_mean is barely different from raw mean.
this is bayesian smoothing (also called "shrinkage"). every recsys uses some version of it. without it, "best movie" gets won by some obscure film with 1 rating of 5.0. imdb's top 250 uses this exact formula. it's also the foundation of how to think about cold-start: a brand new movie's prediction is the global mean, and as ratings come in it gradually shifts.

In [6]:
con.execute(f"""
    SELECT 
        m.title,
        mf.num_ratings,
        ROUND(mf.mean_rating, 3) AS raw_mean,
        ROUND(mf.smoothed_mean, 3) AS smoothed_mean,
        ROUND(mf.mean_rating - mf.smoothed_mean, 3) AS shrinkage
    FROM read_parquet('{movie_features_path}') mf
    JOIN movies m ON m.movieId = mf.movieId
    WHERE mf.num_ratings >= 1000
    ORDER BY mf.smoothed_mean DESC
    LIMIT 10
""").fetchdf()

,title,num_ratings,raw_mean,smoothed_mean,shrinkage
0,Planet Earth II (2016),1119,4.481,4.460,0.021
1,Planet Earth (2006),1743,4.464,4.450,0.013
2,"Shawshank Redemption, The (1994)",81341,4.413,4.413,0.000
3,Band of Brothers (2001),1349,4.395,4.380,0.016
4,"Godfather, The (1972)",52407,4.324,4.323,0.000
5,"Usual Suspects, The (1995)",55287,4.284,4.284,0.000
6,"Godfather: Part II, The (1974)",34119,4.261,4.261,0.001
7,Seven Samurai (Shichinin no samurai) (1954),13345,4.254,4.253,0.001
8,Schindler's List (1993),60323,4.247,4.247,0.000
9,12 Angry Men (1957),16544,4.242,4.241,0.001
